In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys

project_path = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)

from utils.tranformations import reusable

In [0]:
import os

print(os.getcwd())

In [0]:
df = spark.read.format("parquet")\
          .load("abfss://bronze@spotifiproject.dfs.core.windows.net/DimUser") 
          

In [0]:
display(df)

##**DimUser**


###Auto Load

In [0]:
df_user = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/DimUser/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@spotifiproject.dfs.core.windows.net/DimUser")
    

In [0]:
# Static preview — reads directly without streaming
df_preview = spark.read.format("parquet")\
    .load("abfss://bronze@spotifiproject.dfs.core.windows.net/DimUser")
display(df_preview)

In [0]:
display(df_user)

In [0]:
df_user = df_user.withColumn("user_name", upper(col("user_name")))
display(df_user)

In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])
display(df_user)

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@spotifiproject.dfs.core.windows.net/DimUser/data")\
    .toTable("spotifi.silver.DimUser")

##**DimArtist**

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/DimArtist/checkpoint")\
    .option("schemaEvoluationMode", "addNewColumns")\
    .load("abfss://bronze@spotifiproject.dfs.core.windows.net/DimArtist")

In [0]:
display(df_artist)

In [0]:
df_art_obj = reusable()

df_artist = df_art_obj.dropColumns(df_artist,['_rescued_data'])
df_artist = df_artist.dropDuplicates(['artist_id'])
display(df_artist)


In [0]:
df_artist.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@spotifiproject.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotifi.silver.DimArtist")

##**DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/DimTrack/checkpoint")\
    .option("schemaEvoluationMode", "addNewColumns")\
    .load("abfss://bronze@spotifiproject.dfs.core.windows.net/DimTrack")

In [0]:
display(df_track)

In [0]:
df_track_obj = reusable()

df_track = df_track_obj.dropColumns(df_track,['_rescued_data'])
df_track = df_track.dropDuplicates(['track_id'])
display(df_track)

In [0]:
df_track = df_track.withColumn(
    "durationflag",
    when(col("duration_sec") < 150, "low")
        .when(col("duration_sec") < 300, "medium")
        .otherwise("high")
)

df_track = df_track.withColumn(
    "track_name", 
    regexp_replace(col("track_name"),' ','-')
)

display(df_track)

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@spotifiproject.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotifi.silver.DimTrack")

In [0]:
# Static preview — reads directly without streaming
df_preview = spark.read.format("delta")\
    .load("abfss://silver@spotifiproject.dfs.core.windows.net/DimTrack/data")
display(df_preview)

##**DimDate**

In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/DimDate/checkpoint")\
    .option("schemaEvoluationMode", "addNewColumns")\
    .load("abfss://bronze@spotifiproject.dfs.core.windows.net/DimDate")

In [0]:
display(df_date)

In [0]:
df_date = reusable().dropColumns(df_date,['_rescued_data'])

df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@spotifiproject.dfs.core.windows.net/DimDate/data")\
    .toTable("spotifi.silver.DimDate")

##**FactStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/FactStream/checkpoint")\
    .option("schemaEvoluationMode", "addNewColumns")\
    .load("abfss://bronze@spotifiproject.dfs.core.windows.net/FactStream")

In [0]:
display(df_fact)

In [0]:
df_fact = reusable().dropColumns(df_fact,['_rescued_data'])

df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@spotifiproject.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@spotifiproject.dfs.core.windows.net/FactStream/data")\
    .toTable("spotifi.silver.FactStream")